In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# 목표: 반려동물이 입양되는지 예측
# 내 목표는 코드 작동원리를 이해하는가

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers

In [3]:
dataframe = pd.read_csv('datasets/petfinder_mini_extracted/petfinder-mini/petfinder-mini.csv')
dataframe.head()

,Type,Age,Breed1,Gender,Color1,Color2,MaturitySize,FurLength,Vaccinated,Sterilized,Health,Fee,Description,PhotoAmt,AdoptionSpeed
0,Cat,3,Tabby,Male,Black,White,Small,Short,No,No,Healthy,100,Nibble is a 3+ month old ball of cuteness. He ...,1,2
1,Cat,1,Domestic Medium Hair,Male,Black,Brown,Medium,Medium,Not Sure,Not Sure,Healthy,0,I just found it alone yesterday near my apartm...,2,0
2,Dog,1,Mixed Breed,Male,Brown,White,Medium,Medium,Yes,No,Healthy,0,Their pregnant mother was dumped by her irresp...,7,3
3,Dog,4,Mixed Breed,Female,Black,Brown,Medium,Short,Yes,No,Healthy,150,"Good guard dog, very alert, active, obedience ...",8,2
4,Dog,1,Mixed Breed,Male,Black,No Color,Medium,Short,No,No,Healthy,0,This handsome yet cute boy is up for adoption....,3,2


In [4]:
# 표 정리

dataframe['target'] = np.where(dataframe["AdoptionSpeed"]==4, 0, 1)
# np.where(조건, 참일 때 값, 거짓일 때 값)
dataframe = dataframe.drop(columns=['AdoptionSpeed', 'Description'])

dataframe.head()

,Type,Age,Breed1,Gender,Color1,Color2,MaturitySize,FurLength,Vaccinated,Sterilized,Health,Fee,PhotoAmt,target
0,Cat,3,Tabby,Male,Black,White,Small,Short,No,No,Healthy,100,1,1
1,Cat,1,Domestic Medium Hair,Male,Black,Brown,Medium,Medium,Not Sure,Not Sure,Healthy,0,2,1
2,Dog,1,Mixed Breed,Male,Brown,White,Medium,Medium,Yes,No,Healthy,0,7,1
3,Dog,4,Mixed Breed,Female,Black,Brown,Medium,Short,Yes,No,Healthy,150,8,1
4,Dog,1,Mixed Breed,Male,Black,No Color,Medium,Short,No,No,Healthy,0,3,1


In [5]:
train, val, test = np.split(dataframe.sample(frac=1), [int(0.8*(len(dataframe))), int(0.9*len(dataframe))])
# dataframe.sample(frac=1) 나누기전에 섞기.
# (샘플링은 뽑다/ 추출하다 인데 특징이 무작위로 뽑는거라서 셔플이랑 다를 게 없다.(pandas에선 이걸쓰삼. tf에서는 셔플))
# (frac은 비율의 약자. 0.5는 50%, 1은 100%)
# split은 칼자국.

# train = train.reset_index(drop=True) # 인덱스 재정렬
# drop=True는 옛날 번호표들 버려도된다는뜻.

print(len(train))
print(len(val))
print(len(test))

9229
1154
1154


In [6]:
# class로도.. 재조립해보기.

# 데이터셋으로 가공 ! !
def df_to_dataset(dataframe, shuffle=True, batch_size=32):
    df = dataframe.copy()
    labels = df.pop('target')
    df = {key : value.to_numpy()[:, tf.newaxis] for key, value in dataframe.items()}
    
    ds = tf.data.Dataset.from_tensor_slices((dict(df), labels))
    # tf.data.Dataset.from_tensor_slices(문제, 정답) 이렇게 하면 
    # 결과물이 ((나이: 성별: 어쩌구:), 정답:) 이롷게 묶어서 줌. (이게 데이터셋)
    
    # 텐서란. 데이터를 담는 다차원 배열
    # 데이터셋은 사람1데이터들, 사람2데이터들... 
    
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe))
    ds = ds.batch(batch_size)
    ds = ds.prefetch(batch_size)
    return ds

In [7]:
# 확인용 예시
batch_size = 5
train_ds = df_to_dataset(train, batch_size=batch_size)

[(train_features, label_batch)] = train_ds.take(1)
print(list(train_features.keys()))
print(train_features['Age'])
print(label_batch)

['Type', 'Age', 'Breed1', 'Gender', 'Color1', 'Color2', 'MaturitySize', 'FurLength', 'Vaccinated', 'Sterilized', 'Health', 'Fee', 'PhotoAmt', 'target']
tf.Tensor(
[[ 1]
 [ 3]
 [24]
 [ 2]
 [12]], shape=(5, 1), dtype=int64)
tf.Tensor([1 1 1 1 1], shape=(5,), dtype=int64)


2026-03-08 11:01:35.760491: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-03-08 11:01:35.760532: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-03-08 11:01:35.760543: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
2026-03-08 11:01:35.760578: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-08 11:01:35.760596: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [8]:
for x, y in train_ds.take(1):
    print(x)
    print(y)

{'Type': <tf.Tensor: shape=(5, 1), dtype=string, numpy=
array([[b'Cat'],
       [b'Cat'],
       [b'Dog'],
       [b'Dog'],
       [b'Dog']], dtype=object)>, 'Age': <tf.Tensor: shape=(5, 1), dtype=int64, numpy=
array([[12],
       [ 3],
       [ 3],
       [ 1],
       [ 3]])>, 'Breed1': <tf.Tensor: shape=(5, 1), dtype=string, numpy=
array([[b'Domestic Short Hair'],
       [b'Domestic Short Hair'],
       [b'Mixed Breed'],
       [b'Mixed Breed'],
       [b'Mixed Breed']], dtype=object)>, 'Gender': <tf.Tensor: shape=(5, 1), dtype=string, numpy=
array([[b'Male'],
       [b'Female'],
       [b'Male'],
       [b'Female'],
       [b'Female']], dtype=object)>, 'Color1': <tf.Tensor: shape=(5, 1), dtype=string, numpy=
array([[b'Black'],
       [b'Black'],
       [b'Black'],
       [b'Brown'],
       [b'Brown']], dtype=object)>, 'Color2': <tf.Tensor: shape=(5, 1), dtype=string, numpy=
array([[b'No Color'],
       [b'Yellow'],
       [b'White'],
       [b'White'],
       [b'No Color']], dtype=o

이 코드 안에서 아래 네 개의 전처리 레이어를 사용한다.

| Keras 전처리 레이어 | 설명 |
|:---|:---|
| tf.keras.layers.Normalization | 정규화수행. 데이터 분포를 표준화한다. |
| tf.keras.layers.CategoryEncoding | 정수를 원-핫, 멀티-핫 또는 TF-IDF으로 바꾼다. |
| tf.keras.layers.StringLookup | 문자열 범주형 값 -> 정수 인덱스 |
| tf.keras.layers.IntegerLookup | 정수 범주형 값 -> 정수 인덱스 |


In [ ]:
# 숫자열
# 데이터 분포 표준화
def get_normalization_layer(name, dataset):
    
    # 특징 네임텍들이 차원이 있지는 않다.
    nomalizer = layers.Normalization(axis=None)
    
    feature_ds = dataset.map(lambda x, y: x[name])
    
    nomalizer.adapt(feature_ds) # 기준 잡기 (미리 간보기 작업)
    
    return nomalizer

In [10]:
photo_count_col = train_features['PhotoAmt']
layer = get_normalization_layer('PhotoAmt', train_ds)
layer(photo_count_col)

2026-03-08 11:01:36.048885: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


<tf.Tensor: shape=(5, 1), dtype=float32, numpy=
array([[ 0.12694983],
       [-0.19281608],
       [-0.512582  ],
       [-0.19281608],
       [-0.512582  ]], dtype=float32)>

- **숫자 특성이 많은 경우(수백 개 이상) 먼저 숫자 특성을 연결하고 단일 tf.keras.layers.Normalization 레이어를 사용하는 것이 더 효율적이다.**

In [ ]:
# 범주열
# 멀티 핫 인코딩
def get_category_encoding_layer(name, dataset, dtype, max_tokens=None):
    if dtype == 'string':
        index = layers.StringLookup(max_tokens=max_tokens)
    else:
        index = layers.IntegerLookup(max_tokens=max_tokens)
        
    feature_ds = dataset.map(lambda x, y: x[name])
    index.adapt(feature_ds)
    encoder = layers.CategoryEncoding(num_tokens=index.vocabulary_size())
    
    return lambda feature: encoder(index(feature)) 
# 나중에 진짜 데이터 들어오면 인덱스붙이기 -> 인코더 순으로 작업하고 내보내라는 지시서.
# 위 숫자열 할 때에는 한 개의 기계만 사용했으니 lambda사용해서 리턴하지 않아도 됨.
# 연속 동작 지시할 때에는 lambda

In [14]:
test_type_col = train_features['Type']
test_type_layer = get_category_encoding_layer(name='Type',
                                              dataset=train_ds,
                                              dtype='string')
test_type_layer(test_type_col)

<tf.Tensor: shape=(5, 3), dtype=float32, numpy=
array([[0., 1., 0.],
       [0., 0., 1.],
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 1., 0.]], dtype=float32)>

In [15]:
test_age_col = train_features['Age']
test_age_layer = get_category_encoding_layer(name='Age',
                                             dataset=train_ds,
                                             dtype='int64',
                                             max_tokens=5)
test_age_layer(test_age_col)

<tf.Tensor: shape=(5, 5), dtype=float32, numpy=
array([[0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0.]], dtype=float32)>

In [ ]:
# class 고안..?
